# GameVault GPU Model Trainer v1.1 (Steam 41M + 2026 Tags + IGDB v4)

High-accuracy, zero-leakage offline recommendation pipeline optimized for **Google Colab L4 GPU (~1.94 CU/hr, 22.5 GB VRAM, ~7-9 mins total)** — also supports A100 / Blackwell / T4.

### Before Clicking "Run All"
1. **Runtime -> Change runtime type** -> Select **L4 GPU** (best cost/speed: fast Tensor/CUDA FP32 `EASE^R` + ~2 min `bge-large` FP16 encoding; A100 uses native FP64 automatically).
2. Open the **Secrets (key icon)** tab in Colab's left sidebar and add:
   - `IGDB_CLIENT_ID`
   - `IGDB_CLIENT_SECRET`
   *(Toggle "Notebook access" ON for both!)*
3. Click **Runtime -> Run all**.
4. At the end, Colab downloads **`game_models_bundle.zip`** (~15 MB unzipped). Unzip into `public/models/games/` in your repo:
   - `game_ease_k80.cvea` (Full-rank source- & target-debiased `EASE^R` graph, `CVEA v1`, ~5.6 MB)
   - `game_embeddings_f16.bin` (Winning collaborative 64-D embeddings: Debiased SVD vs Positive-Eigen Spectral-EASE vs Freq-Reg GPU CG-iALS, `CVEM v1`, ~2.3 MB)
   - `game_content_f16.bin` (Tag-dropout collaborative-aligned `BAAI/bge-large-en-v1.5` `1024-D -> 128-D` float16 across ~26.5k IGDB games incl. PS/Nintendo exclusives & 2023-2026 releases, `CVEM v1`, ~6.5 MB)
   - `game_ranker_weights.json` (Dev-500 tuned / Holdout-500 verified blend weights + bootstrap 95% CIs + true cold-start metrics)
   - `game_tuner_axes.json` (Popularity-decorrelated `Intensity`/`Pacing` & `Tone` 128-D mood vectors, compatible with `tunerService.js`)
   - `game_steam_to_igdb.json` (Canonical Steam `app_id -> igdb_id` crosswalk for instant 0-API Steam Library imports)

In [ ]:
# Install only missing/lightweight packages (do NOT -U upgrade pandas/scipy in a live Colab kernel!)
!pip -q install kagglehub datasets sentence-transformers huggingface_hub pyarrow requests tqdm

import gc
import json
import math
import os
import random
import re
import struct
import time
import zipfile
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
from scipy.sparse.linalg import svds
import torch
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Hardware detection: L4 (Ada, 22.5 GiB) has 1:64 throttled FP64 (~0.47 TFLOPS vs 30.3 TFLOPS FP32).
# Only enable FP64 Cholesky on data-center GPUs with native high-throughput FP64 cores!
assert torch.cuda.is_available(), "No GPU detected! Switch Runtime -> Change runtime type -> L4 GPU (or A100 / T4)."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
HAS_NATIVE_FP64 = any(tag in GPU_NAME.upper() for tag in ("A100", "H100", "H200", "B100", "B200", "V100"))
USE_FP64_SOLVER = HAS_NATIVE_FP64 and VRAM_GB >= 22.0
print(
    f"GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GB | pandas: {pd.__version__} | "
    f"EASE solver dtype: {'float64 (native HPC)' if USE_FP64_SOLVER else 'float32 (fast Tensor/CUDA)'}"
)

# Load IGDB credentials from Colab Secrets (or env / interactive prompt)
def load_secret(name: str) -> str:
    val = os.environ.get(name, "").strip()
    if not val:
        try:
            from google.colab import userdata
            val = (userdata.get(name) or "").strip()
        except Exception:
            val = ""
    if not val:
        val = getpass(f"Enter {name}: ").strip()
    if not val:
        raise RuntimeError(f"Missing required secret: {name}")
    return val

IGDB_CLIENT_ID = load_secret("IGDB_CLIENT_ID")
IGDB_CLIENT_SECRET = load_secret("IGDB_CLIENT_SECRET")
print("IGDB credentials loaded successfully (values hidden).")

# Pipeline directories (automatically uses mounted Google Drive cache if available so reconnects skip IGDB calls)
DRIVE_MOUNT = Path("/content/drive/MyDrive")
if DRIVE_MOUNT.exists():
    WORK_DIR = DRIVE_MOUNT / "gamevault_colab_cache"
    print(f"Using persistent Google Drive cache at: {WORK_DIR}")
else:
    WORK_DIR = Path("/content/game_work")
OUT_DIR = Path("/content/game_models_out")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Steam dataset filters
    "min_steam_app_reviews_for_crosswalk": 25,  # Map Steam apps with >= 25 reviews (skips 15-review shovelware)
    "min_game_reviews": 40,                     # Keep games with >= 40 mapped reviews in collab matrix
    "max_collab_games": 18000,                  # Cap collaborative matrix width (<65535 uint16 CVEA limit, ~5.6 MB)
    "min_user_reviews": 5,                      # Drop 1-4 review accounts to eliminate noise
    # IGDB catalog with explicit reserved quotas (prevents Steam tail from crowding out PS/Nintendo & 2023-2026!)
    "max_steam_catalog_core": 18000,            # Top 18k Steam-matched games by Steam review count
    "exclusives_and_classics_quota": 5500,      # Dedicated budget for PS/Nintendo/Xbox exclusives & classics
    "recent_2023_2026_quota": 3000,             # Dedicated budget for 2023-2026 released hits + hyped titles
    "max_igdb_catalog": 26500,                  # Total content index cap (~6.5 MB float16 CVEM v1)
    "valid_game_types": {0, 4, 8, 9, 10},       # Main(0), Standalone Exp(4), Remake(8), Remaster(9), Expanded(10)
    # Evaluation split: 500 Dev (for tuning) + 500 Untouched Holdout Test (for final scorecard)
    "eval_users": 1000,
    "eval_min_positives": 20,
    "eval_min_train_positives": 10,             # Guard against same-day backlog review wipes
    "eval_holdout_k": 5,                        # Hold out last 5 positive games chronologically
    # Output binary dimensions (100% compatible with CinemaVault CVEA v1 & CVEM v1 loaders)
    "ease_k": 80,
    "collab_dim": 64,
    "content_dim": 128,
    "bge_model": "BAAI/bge-large-en-v1.5",
}
print("Config:", json.dumps({k: (list(v) if isinstance(v, set) else v) for k, v in CONFIG.items()}, indent=2))

In [ ]:
import kagglehub

t0 = time.time()
DATA_DIR = Path(kagglehub.dataset_download("antonkozyriev/game-recommendations-on-steam"))
print(f"Dataset ready at {DATA_DIR} in {time.time() - t0:.1f}s")

for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / (1024**2):8.2f} MB")

games_df = pd.read_csv(DATA_DIR / "games.csv")
print(f"\ngames.csv: {len(games_df):,} rows | columns: {list(games_df.columns)}")
steam_app_reviews_map = dict(
    zip(
        games_df["app_id"].dropna().astype(int).to_numpy(),
        games_df["user_reviews"].fillna(0).astype(int).to_numpy(),
    )
)

BOILERPLATE_TAGS = {
    "steam achievements", "steam cloud", "steam trading cards", "steam leaderboards",
    "steam workshop", "full controller support", "partial controller support",
    "family sharing", "remote play on phone", "remote play on tablet", "remote play on tv",
    "remote play together", "in-app purchases", "includes level editor", "captions available",
    "commentary available", "stats", "valve anti-cheat enabled", "hdr available",
    "tracked controller support", "vr supported", "vr only", "camera comfort",
    "custom volume controls", "adjustable difficulty", "playable without timed input",
    "stereo sound", "surround sound", "subtitle options", "color alternatives",
    "mouse only option", "touch only option", "keyboard only option", "save anytime",
    "singleplayer", "single-player", "multiplayer", "multi-player", "exclusive", "true exclusive",
}

def clean_tag_list(raw_list, limit: int = 18) -> list[str]:
    out, seen = [], set()
    for t in raw_list:
        s = str(t).strip()
        sl = s.lower()
        if not s or sl in BOILERPLATE_TAGS or sl in seen:
            continue
        seen.add(sl)
        out.append(s)
        if len(out) >= limit:
            break
    return out

# 1. Load games_metadata.json (JSON Lines: app_id, description, tags)
steam_meta = {}
meta_path = DATA_DIR / "games_metadata.json"
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                aid = int(obj["app_id"])
                steam_meta[aid] = {
                    "description": (obj.get("description") or "").strip(),
                    "tags": clean_tag_list(obj.get("tags") or [], limit=18),
                }
            except Exception:
                continue
print(f"Loaded antonkozyriev Steam metadata/tags for {len(steam_meta):,} apps.")

# 2. Enrich with FronkonGames/steam-games-dataset (2026 store descriptions, genres & any community tags)
# Verified HF schema: appID (camelCase!), short_description, detailed_description, genres, categories, tags
def parse_fg_tags(raw_tags) -> list[str]:
    if raw_tags is None:
        return []
    if isinstance(raw_tags, dict):
        items = sorted(raw_tags.items(), key=lambda kv: float(kv[1] or 0), reverse=True)
        return [str(k).strip() for k, _ in items if str(k).strip()]
    if isinstance(raw_tags, (list, tuple, np.ndarray)):
        return [str(t).strip() for t in raw_tags if str(t).strip()]
    if isinstance(raw_tags, str):
        s = raw_tags.strip()
        if not s:
            return []
        if s.startswith("{") or s.startswith("["):
            try:
                return parse_fg_tags(json.loads(s.replace("'", '"')))
            except Exception:
                pass
        sep = "|" if "|" in s else ","
        return [x.strip() for x in s.split(sep) if x.strip()]
    return []

try:
    t_fg = time.time()
    try:
        fg_df = pd.read_parquet(
            "hf://datasets/FronkonGames/steam-games-dataset/data/train-00000-of-00001.parquet",
            columns=["appID", "short_description", "detailed_description", "genres", "categories", "tags"],
        )
        fg_iter = fg_df.to_dict("records")
    except Exception:
        from datasets import load_dataset
        fg_iter = load_dataset("FronkonGames/steam-games-dataset", split="train")

    fg_added, fg_desc_upgraded, fg_tags_upgraded = 0, 0, 0
    for row in fg_iter:
        raw_id = row.get("appID") if row.get("appID") is not None else (
            row.get("AppID") or row.get("app_id") or row.get("appid")
        )
        if raw_id is None:
            continue
        try:
            aid = int(raw_id)
        except Exception:
            continue

        raw_t = parse_fg_tags(row.get("tags") if row.get("tags") is not None else row.get("Tags"))
        raw_g = parse_fg_tags(row.get("genres") if row.get("genres") is not None else row.get("Genres"))
        tags = clean_tag_list(raw_t + raw_g, limit=18)
        desc = (
            row.get("short_description")
            or row.get("detailed_description")
            or row.get("About the game")
            or row.get("about_the_game")
            or ""
        )
        desc = str(desc).strip()[:600]
        if not tags and not desc:
            continue

        if aid not in steam_meta:
            steam_meta[aid] = {"description": desc, "tags": tags}
            fg_added += 1
        else:
            if len(tags) > len(steam_meta[aid]["tags"]):
                steam_meta[aid]["tags"] = tags
                fg_tags_upgraded += 1
            if not steam_meta[aid]["description"] and desc:
                steam_meta[aid]["description"] = desc
                fg_desc_upgraded += 1

    del fg_iter
    gc.collect()
    assert (fg_added + fg_desc_upgraded + fg_tags_upgraded) > 0, "FronkonGames matched 0 rows — check column names!"
    print(
        f"FronkonGames 2026 merged in {time.time() - t_fg:.1f}s: "
        f"+{fg_added:,} new Steam apps, {fg_tags_upgraded:,} tag lists upgraded, "
        f"{fg_desc_upgraded:,} descriptions filled (total {len(steam_meta):,} Steam apps)"
    )
except Exception as e:
    print(f"FronkonGames enrichment warning ({e}); continuing with base Steam metadata.")

# 3. Pull multi-platform 2024-2026 gameplay tags via fast 4-column Parquet projection (~15 MB)
# Restores rich tags ("Roguelike Deckbuilder", "Souls-like", "Dark Fantasy") for 2024-2026 hits & console exclusives!
def norm_title_key(title: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(title or "").lower())

multiplatform_tags_by_title = {}
try:
    t_mp = time.time()
    mp_df = pd.read_parquet(
        "hf://datasets/atalaydenknalbant/video-games-dataset/videogames_data.parquet",
        columns=["name", "ratings_count", "tags"],
    )
    mp_df = mp_df.dropna(subset=["name", "tags"])
    mp_df["ratings_count"] = mp_df["ratings_count"].fillna(0).astype(int)
    mp_df = mp_df.sort_values("ratings_count", ascending=False)
    for nm, tg_str in zip(mp_df["name"].to_numpy(), mp_df["tags"].to_numpy()):
        key = norm_title_key(nm)
        if not key or key in multiplatform_tags_by_title:
            continue
        cleaned = clean_tag_list(parse_fg_tags(tg_str), limit=15)
        if cleaned:
            multiplatform_tags_by_title[key] = cleaned
    del mp_df
    gc.collect()
    print(f"Loaded multi-platform 2026 gameplay tags for {len(multiplatform_tags_by_title):,} titles in {time.time() - t_mp:.1f}s")
except Exception as e:
    print(f"Optional multi-platform tag enrichment skipped ({e}).")

In [ ]:
class IGDBClient:
    def __init__(self, client_id: str, client_secret: str):
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None
        self.token_expiry = 0.0
        self.last_req = 0.0
        self.session = requests.Session()
        self._refresh_token()

    def _refresh_token(self):
        r = self.session.post(
            "https://id.twitch.tv/oauth2/token",
            data={
                "client_id": self.client_id,
                "client_secret": self.client_secret,
                "grant_type": "client_credentials",
            },
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()
        self.token = data["access_token"]
        self.token_expiry = time.time() + max(60, int(data.get("expires_in", 3600)) - 120)

    def query(self, endpoint: str, body: str, max_retries: int = 5):
        for attempt in range(max_retries):
            if time.time() >= self.token_expiry:
                self._refresh_token()
            # Respect IGDB 4 req/sec rate limit cleanly
            elapsed = time.time() - self.last_req
            if elapsed < 0.265:
                time.sleep(0.265 - elapsed)
            self.last_req = time.time()
            try:
                r = self.session.post(
                    f"https://api.igdb.com/v4/{endpoint}",
                    headers={
                        "Client-ID": self.client_id,
                        "Authorization": f"Bearer {self.token}",
                    },
                    data=body.encode("utf-8"),
                    timeout=30,
                )
                if r.status_code == 200:
                    return r.json()
                if r.status_code == 401:
                    self._refresh_token()
                    continue
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(1.0 * (2 ** attempt))
                    continue
                raise RuntimeError(f"IGDB {endpoint} HTTP {r.status_code}: {r.text[:250]}")
            except requests.RequestException:
                if attempt == max_retries - 1:
                    raise
                time.sleep(1.5 * (attempt + 1))
        raise RuntimeError(f"IGDB {endpoint} failed after {max_retries} retries")

igdb = IGDBClient(IGDB_CLIENT_ID, IGDB_CLIENT_SECRET)

CROSSWALK_CACHE = WORK_DIR / "steam_to_igdb_v1_1.json"
CATALOG_CACHE = WORK_DIR / "igdb_catalog_v1_1.json"

GAME_FIELDS = (
    "id,name,game_type,version_parent,first_release_date,total_rating,total_rating_count,rating_count,hypes,"
    "genres.name,themes.name,player_perspectives.name,game_modes.name,keywords.name,"
    "franchises.name,collections.name,platforms.name,"
    "involved_companies.company.name,involved_companies.developer,involved_companies.publisher,"
    "summary,storyline"
)

def normalize_igdb_game(g: dict) -> dict:
    devs, pubs = [], []
    for ic in g.get("involved_companies") or []:
        cname = (ic.get("company") or {}).get("name")
        if not cname:
            continue
        if ic.get("developer"):
            devs.append(cname)
        elif ic.get("publisher"):
            pubs.append(cname)
    ts = g.get("first_release_date")
    year = int(pd.to_datetime(ts, unit="s", utc=True).year) if ts else None
    return {
        "id": int(g["id"]),
        "name": (g.get("name") or "").strip(),
        "game_type": int(g.get("game_type") if g.get("game_type") is not None else 0),
        "version_parent": int(g["version_parent"]) if g.get("version_parent") is not None else None,
        "year": year,
        "total_rating": float(g["total_rating"]) if g.get("total_rating") is not None else None,
        "total_rating_count": int(g.get("total_rating_count") or 0),
        "rating_count": int(g.get("rating_count") or 0),
        "hypes": int(g.get("hypes") or 0),
        "genres": [x["name"] for x in (g.get("genres") or []) if x.get("name")],
        "themes": [x["name"] for x in (g.get("themes") or []) if x.get("name")],
        "perspectives": [x["name"] for x in (g.get("player_perspectives") or []) if x.get("name")],
        "modes": [x["name"] for x in (g.get("game_modes") or []) if x.get("name")],
        "keywords": [x["name"] for x in (g.get("keywords") or []) if x.get("name")][:20],
        "franchises": [x["name"] for x in (g.get("franchises") or []) if x.get("name")],
        "collections": [x["name"] for x in (g.get("collections") or []) if x.get("name")],
        "platforms": [x["name"] for x in (g.get("platforms") or []) if x.get("name")][:12],
        "developers": devs[:4],
        "publishers": pubs[:3],
        "summary": (g.get("summary") or "").strip(),
        "storyline": (g.get("storyline") or "").strip(),
    }

# Step 4A: Steam app_id -> IGDB id crosswalk
if CROSSWALK_CACHE.exists():
    raw_cw = json.loads(CROSSWALK_CACHE.read_text(encoding="utf-8"))
    raw_steam_to_igdb = {int(k): int(v) for k, v in raw_cw.items()}
    print(f"Loaded cached Steam->IGDB crosswalk: {len(raw_steam_to_igdb):,} mappings")
else:
    cand_apps = (
        games_df[games_df["user_reviews"] >= CONFIG["min_steam_app_reviews_for_crosswalk"]]
        .sort_values("user_reviews", ascending=False)["app_id"]
        .dropna().astype(int).unique().tolist()
    )
    print(f"Querying IGDB external_games for {len(cand_apps):,} Steam apps (>= {CONFIG['min_steam_app_reviews_for_crosswalk']} reviews)...")
    raw_steam_to_igdb = {}
    batch_size = 200
    for i in tqdm(range(0, len(cand_apps), batch_size), desc="Steam->IGDB crosswalk"):
        batch = cand_apps[i : i + batch_size]
        uids = ",".join(f'"{aid}"' for aid in batch)
        rows = igdb.query(
            "external_games",
            f"fields game,uid; where uid = ({uids}) & external_game_source = 1; limit 500;",
        )
        for row in rows:
            uid_str = str(row.get("uid", "")).strip()
            gid = row.get("game")
            if uid_str.isdigit() and gid:
                raw_steam_to_igdb[int(uid_str)] = int(gid)
    CROSSWALK_CACHE.write_text(json.dumps(raw_steam_to_igdb), encoding="utf-8")
    print(f"Mapped {len(raw_steam_to_igdb):,} / {len(cand_apps):,} Steam apps to IGDB IDs.")

# Step 4B + 4C: Build IGDB Catalog with Explicit Reserved Quotas + GOTY/Deluxe Edition Remapping!
if CATALOG_CACHE.exists():
    cached_bundle = json.loads(CATALOG_CACHE.read_text(encoding="utf-8"))
    igdb_catalog = {int(k): v for k, v in cached_bundle["catalog"].items()}
    edition_remap = {int(k): int(v) for k, v in cached_bundle.get("edition_remap", {}).items()}
    for aid, gid in list(raw_steam_to_igdb.items()):
        if gid in edition_remap:
            raw_steam_to_igdb[aid] = edition_remap[gid]
    print(f"Loaded cached IGDB catalog: {len(igdb_catalog):,} canonical games ({len(edition_remap):,} editions remapped)")
else:
    # Aggregate Steam review support per IGDB ID so top Steam games enter the 18k core first
    igdb_steam_support = {}
    for aid, gid in raw_steam_to_igdb.items():
        igdb_steam_support[gid] = igdb_steam_support.get(gid, 0) + steam_app_reviews_map.get(aid, 0)
    matched_igdb_ids = sorted(igdb_steam_support.keys(), key=lambda g: igdb_steam_support[g], reverse=True)

    print(f"Fetching IGDB metadata for {len(matched_igdb_ids):,} Steam-matched IDs (with edition remapping)...")
    fetched_steam_games = {}
    edition_remap = {}
    parent_ids_needed = set()

    for i in tqdm(range(0, len(matched_igdb_ids), 400), desc="IGDB Steam-matched"):
        batch = matched_igdb_ids[i : i + 400]
        id_csv = ",".join(str(x) for x in batch)
        rows = igdb.query("games", f"fields {GAME_FIELDS}; where id = ({id_csv}); limit 500;")
        for g in rows:
            gid = int(g["id"])
            vp = g.get("version_parent")
            if vp is not None:
                # Remap GOTY / Deluxe / Collector's Edition SKUs to their canonical base game!
                edition_remap[gid] = int(vp)
                parent_ids_needed.add(int(vp))
                continue
            gt = int(g.get("game_type") if g.get("game_type") is not None else 0)
            if gt in CONFIG["valid_game_types"] and g.get("name"):
                fetched_steam_games[gid] = normalize_igdb_game(g)

    missing_parents = sorted(pid for pid in parent_ids_needed if pid not in fetched_steam_games)
    if missing_parents:
        for i in range(0, len(missing_parents), 400):
            batch = missing_parents[i : i + 400]
            id_csv = ",".join(str(x) for x in batch)
            rows = igdb.query("games", f"fields {GAME_FIELDS}; where id = ({id_csv}) & version_parent = null; limit 500;")
            for g in rows:
                gt = int(g.get("game_type") if g.get("game_type") is not None else 0)
                if gt in CONFIG["valid_game_types"] and g.get("name"):
                    fetched_steam_games[int(g["id"])] = normalize_igdb_game(g)

    # Apply edition -> canonical parent remap to raw_steam_to_igdb so reviews merge onto the base game!
    for aid, gid in list(raw_steam_to_igdb.items()):
        if gid in edition_remap:
            raw_steam_to_igdb[aid] = edition_remap[gid]
            igdb_steam_support[edition_remap[gid]] = igdb_steam_support.get(edition_remap[gid], 0) + steam_app_reviews_map.get(aid, 0)

    # Keep top max_steam_catalog_core (18,000) Steam games in igdb_catalog so console exclusives NEVER starve!
    ranked_steam_gids = sorted(fetched_steam_games.keys(), key=lambda g: igdb_steam_support.get(g, 0), reverse=True)
    igdb_catalog = {gid: fetched_steam_games[gid] for gid in ranked_steam_gids[:CONFIG["max_steam_catalog_core"]]}
    del fetched_steam_games
    print(f"Canonical Steam core locked: {len(igdb_catalog):,} games ({len(edition_remap):,} edition SKUs merged into base games).")

    valid_types_str = ",".join(str(x) for x in sorted(CONFIG["valid_game_types"]))

    # Quota 2: Top overall IGDB Classics & Console Exclusives (PS / Nintendo / Xbox) — guaranteed +5,500 budget!
    target_after_exclusives = min(
        CONFIG["max_igdb_catalog"] - CONFIG["recent_2023_2026_quota"],
        len(igdb_catalog) + CONFIG["exclusives_and_classics_quota"],
    )
    offset = 0
    pbar = tqdm(total=max(0, target_after_exclusives - len(igdb_catalog)), desc="IGDB Console Exclusives & Classics")
    while len(igdb_catalog) < target_after_exclusives and offset < 25000:
        rows = igdb.query(
            "games",
            f"fields {GAME_FIELDS}; "
            f"where game_type = ({valid_types_str}) & version_parent = null & total_rating_count >= 5; "
            f"sort total_rating_count desc; limit 500; offset {offset};",
        )
        if not rows:
            break
        added = 0
        for g in rows:
            gid = int(g["id"])
            if gid not in igdb_catalog and g.get("name"):
                igdb_catalog[gid] = normalize_igdb_game(g)
                added += 1
                if len(igdb_catalog) >= target_after_exclusives:
                    break
        pbar.update(added)
        offset += 500
        if len(rows) < 500:
            break
    pbar.close()
    print(f"Catalog after Exclusives/Classics pass: {len(igdb_catalog):,} games.")

    # Quota 3A: Released 2023-2026 acclaimed & popular games FIRST (guarantees Balatro, Wukong, Astro Bot, Expedition 33, Hades II!)
    ts_2023 = int(pd.Timestamp("2023-01-01", tz="UTC").timestamp())
    ts_now = int(time.time()) + 86400 * 30
    offset = 0
    while len(igdb_catalog) < CONFIG["max_igdb_catalog"] - 400 and offset < 10000:
        rows = igdb.query(
            "games",
            f"fields {GAME_FIELDS}; "
            f"where game_type = ({valid_types_str}) & version_parent = null "
            f"& first_release_date >= {ts_2023} & first_release_date <= {ts_now} & total_rating_count >= 3; "
            f"sort total_rating_count desc; limit 500; offset {offset};",
        )
        if not rows:
            break
        for g in rows:
            gid = int(g["id"])
            if gid not in igdb_catalog and g.get("name"):
                igdb_catalog[gid] = normalize_igdb_game(g)
                if len(igdb_catalog) >= CONFIG["max_igdb_catalog"] - 400:
                    break
        offset += 500
        if len(rows) < 500:
            break

    # Quota 3B: Top hyped recent/upcoming 2023-2026 titles for remaining ~400 slots
    offset = 0
    while len(igdb_catalog) < CONFIG["max_igdb_catalog"] and offset < 3000:
        rows = igdb.query(
            "games",
            f"fields {GAME_FIELDS}; "
            f"where game_type = ({valid_types_str}) & version_parent = null & first_release_date >= {ts_2023} & hypes >= 15; "
            f"sort hypes desc; limit 500; offset {offset};",
        )
        if not rows:
            break
        for g in rows:
            gid = int(g["id"])
            if gid not in igdb_catalog and g.get("name"):
                igdb_catalog[gid] = normalize_igdb_game(g)
                if len(igdb_catalog) >= CONFIG["max_igdb_catalog"]:
                    break
        offset += 500
        if len(rows) < 500:
            break

    CATALOG_CACHE.write_text(json.dumps({"catalog": igdb_catalog, "edition_remap": edition_remap}), encoding="utf-8")
    print(f"Saved canonical IGDB catalog: {len(igdb_catalog):,} valid games.")

# Final verified crosswalk: keep Steam apps whose canonical IGDB game is in igdb_catalog
steam_to_igdb = {aid: gid for aid, gid in raw_steam_to_igdb.items() if gid in igdb_catalog}

igdb_to_steam_apps = {}
for aid, gid in steam_to_igdb.items():
    igdb_to_steam_apps.setdefault(gid, []).append(aid)

# Reverse-lookup Steam app_ids for unlinked IGDB games (batch_size=150 so multi-SKU games never hit limit 500!)
unlinked_igdb_ids = [gid for gid in igdb_catalog.keys() if gid not in igdb_to_steam_apps]
if unlinked_igdb_ids:
    REV_CACHE = WORK_DIR / "igdb_reverse_steam_v1_1.json"
    if REV_CACHE.exists():
        rev_map = {int(k): v for k, v in json.loads(REV_CACHE.read_text(encoding="utf-8")).items()}
    else:
        rev_map = {}
        print(f"Reverse-linking {len(unlinked_igdb_ids):,} additional IGDB games (batch=150) to Steam app_ids...")
        for i in tqdm(range(0, len(unlinked_igdb_ids), 150), desc="IGDB->Steam reverse lookup"):
            batch = unlinked_igdb_ids[i : i + 150]
            id_csv = ",".join(str(x) for x in batch)
            rows = igdb.query(
                "external_games",
                f"fields game,uid; where game = ({id_csv}) & external_game_source = 1; limit 500;",
            )
            for row in rows:
                uid_str = str(row.get("uid", "")).strip()
                gid = row.get("game")
                if uid_str.isdigit() and gid:
                    rev_map.setdefault(int(gid), []).append(int(uid_str))
        REV_CACHE.write_text(json.dumps(rev_map), encoding="utf-8")
    for gid, aids in rev_map.items():
        igdb_to_steam_apps.setdefault(gid, []).extend(aids)
    print(f"Reverse-linked {len(rev_map):,} extra IGDB games to Steam app_ids.")

# Attach Steam tags + multi-platform 2026 gameplay tag fallback onto IGDB catalog entries
steam_tags_attached, mp_tags_attached = 0, 0
for gid, item in igdb_catalog.items():
    tags, seen_t = [], set()
    steam_desc = ""
    for aid in igdb_to_steam_apps.get(gid, []):
        m = steam_meta.get(aid)
        if not m:
            continue
        if not steam_desc and m["description"]:
            steam_desc = m["description"]
        for t in m["tags"]:
            tl = t.lower()
            if tl not in seen_t:
                seen_t.add(tl)
                tags.append(t)
    if tags:
        steam_tags_attached += 1
    else:
        mp_fallback = multiplatform_tags_by_title.get(norm_title_key(item["name"]), [])
        if mp_fallback:
            tags = list(mp_fallback)
            mp_tags_attached += 1
    item["steam_tags"] = tags[:18]
    item["steam_desc"] = steam_desc

print(f"Verified Steam->IGDB mappings: {len(steam_to_igdb):,}")
print(
    f"IGDB catalog total: {len(igdb_catalog):,} "
    f"({steam_tags_attached:,} w/ Steam tags + {mp_tags_attached:,} w/ 2026 multi-platform tags)"
)

# Verify console exclusives AND 2024-2026 hits are present in catalog!
for probe in [
    "Bloodborne",
    "The Legend of Zelda: Tears of the Kingdom",
    "Super Mario Odyssey",
    "Astro Bot",
    "Black Myth: Wukong",
    "Balatro",
    "Elden Ring",
    "Persona 5 Royal",
]:
    hits = [g for g in igdb_catalog.values() if g["name"].lower() == probe.lower()]
    status = (
        f"FOUND (id={hits[0]['id']}, Steam={'yes' if hits[0]['id'] in igdb_to_steam_apps else 'NO - exclusive'}, tags={len(hits[0]['steam_tags'])})"
        if hits else "MISSING!"
    )
    print(f"  Check '{probe}': {status}")

In [ ]:
# Load users.csv into compact sorted NumPy arrays (~115 MB RAM vs ~1.6 GB Python dict!)
users_df = pd.read_csv(
    DATA_DIR / "users.csv",
    usecols=["user_id", "reviews"],
    dtype={"user_id": np.int64, "reviews": np.int32},
)
users_df.sort_values("user_id", kind="quicksort", inplace=True)
u_sorted_ids = users_df["user_id"].to_numpy()
u_sorted_revs = users_df["reviews"].to_numpy()
del users_df
gc.collect()

# Fast vectorized Steam app_id -> IGDB id lookup array
assert len(steam_to_igdb) > 0, "steam_to_igdb is empty!"
max_app_id = max(steam_to_igdb.keys()) + 1
app_to_igdb_arr = np.zeros(max_app_id, dtype=np.int32)
for aid, gid in steam_to_igdb.items():
    app_to_igdb_arr[aid] = gid

# Stream recommendations.csv in 2M-row chunks (Pandas 1.x/2.x/3.x safe datetime64[D] + NaN playtime guard!)
rec_path = DATA_DIR / "recommendations.csv"
chunks = []
total_raw = 0
total_kept = 0
t0 = time.time()

reader = pd.read_csv(
    rec_path,
    usecols=["app_id", "user_id", "is_recommended", "hours", "date"],
    dtype={"app_id": np.int32, "user_id": np.int64, "is_recommended": bool, "hours": np.float32},
    chunksize=2_000_000,
)

for chunk in tqdm(reader, desc="Streaming 41M Steam reviews"):
    total_raw += len(chunk)
    aids = chunk["app_id"].to_numpy()
    in_bounds = (aids >= 0) & (aids < max_app_id)
    gids = np.zeros(len(aids), dtype=np.int32)
    gids[in_bounds] = app_to_igdb_arr[aids[in_bounds]]
    mask = gids > 0
    if not np.any(mask):
        continue

    raw_hours = np.nan_to_num(chunk["hours"].to_numpy()[mask], nan=0.0, posinf=5000.0, neginf=0.0)
    dt = pd.to_datetime(chunk["date"].to_numpy()[mask], errors="coerce")
    valid_dt = ~pd.isna(dt)
    days = np.zeros(len(dt), dtype=np.int32)
    if np.any(valid_dt):
        days[valid_dt] = dt[valid_dt].to_numpy(dtype="datetime64[D]").astype(np.int32)

    sub = pd.DataFrame({
        "user_id": chunk["user_id"].to_numpy()[mask],
        "igdb_id": gids[mask],
        "is_rec": chunk["is_recommended"].to_numpy()[mask],
        "hours": np.clip(raw_hours, 0.0, 5000.0).astype(np.float32),
        "day": days,
    })
    chunks.append(sub)
    total_kept += len(sub)

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

# Sanity-check date resolution so Pandas unit changes can NEVER silently collapse days!
day_span = int(df["day"].max()) - int(df.loc[df["day"] > 0, "day"].min())
assert day_span > 2000, f"Date parsing unit bug detected: day_span={day_span} days (expected >2000)!"
print(f"Streamed {total_raw:,} rows -> kept {total_kept:,} valid IGDB game reviews in {time.time() - t0:.1f}s (span={day_span:,} days)")

# Deduplicate (user_id, igdb_id) keeping the most recent review (merges base + GOTY/edition Steam SKUs cleanly)
df.sort_values("day", kind="mergesort", inplace=True)
df.drop_duplicates(subset=["user_id", "igdb_id"], keep="last", inplace=True)

# Iterative 2-pass core filter
for _ in range(2):
    g_counts = df["igdb_id"].value_counts()
    valid_games = g_counts[g_counts >= CONFIG["min_game_reviews"]].head(CONFIG["max_collab_games"]).index
    df = df[df["igdb_id"].isin(valid_games)]
    u_counts = df["user_id"].value_counts()
    valid_users = u_counts[u_counts >= CONFIG["min_user_reviews"]].index
    df = df[df["user_id"].isin(valid_users)]

df.reset_index(drop=True, inplace=True)

# Dense contiguous item & user indices (sorted IGDB IDs, <= 18,000 fits CVEA uint16!)
collab_igdb_ids = np.sort(df["igdb_id"].unique()).astype(np.uint32)
igdb_to_idx = {int(gid): i for i, gid in enumerate(collab_igdb_ids)}
n_items = len(collab_igdb_ids)
assert n_items <= 65535, f"n_items={n_items} exceeds CVEA v1 uint16 limit!"

user_codes, unique_users = pd.factorize(df["user_id"], sort=False)
n_users = len(unique_users)
df["u_idx"] = user_codes.astype(np.int32)
df["i_idx"] = df["igdb_id"].map(igdb_to_idx).astype(np.int32)

# Vectorized power-reviewer lookup via searchsorted (dampens 1,000+ review curator/bot accounts)
u_arr = unique_users.to_numpy(dtype=np.int64)
pos_idx = np.searchsorted(u_sorted_ids, u_arr)
pos_idx_clipped = np.clip(pos_idx, 0, max(0, len(u_sorted_ids) - 1))
matched_u = (len(u_sorted_ids) > 0) & (u_sorted_ids[pos_idx_clipped] == u_arr)
u_tot_revs = np.where(matched_u, u_sorted_revs[pos_idx_clipped], 10).astype(np.float32)
del u_sorted_ids, u_sorted_revs
gc.collect()

user_damping = 1.0 / np.log2(2.0 + np.maximum(0.0, u_tot_revs - 25.0) / 25.0)
user_damping = np.clip(user_damping, 0.25, 1.0).astype(np.float32)

print(f"Final Collaborative Core: {len(df):,} reviews | {n_users:,} users | {n_items:,} IGDB games")

# Strict Chronological Holdout: sample candidate pool, require >=10 past train positives AND >=3 future positives!
pos_df = df[df["is_rec"]]
pos_u_counts = pos_df["u_idx"].value_counts()
eligible_eval_users = pos_u_counts[pos_u_counts >= CONFIG["eval_min_positives"]].index.to_numpy()
rng = np.random.default_rng(SEED)
shuffled_candidates = rng.permutation(eligible_eval_users)[: min(len(eligible_eval_users), CONFIG["eval_users"] * 3)]
cand_set = set(int(u) for u in shuffled_candidates)

eval_sub = df[df["u_idx"].isin(cand_set)].sort_values(["u_idx", "day"])
grouped_cand = {int(u): grp for u, grp in eval_sub.groupby("u_idx", sort=False)}

holdout_targets = {}
drop_mask = np.zeros(len(df), dtype=bool)
for u in shuffled_candidates:
    u_int = int(u)
    grp = grouped_cand.get(u_int)
    if grp is None:
        continue
    pos_grp = grp[grp["is_rec"]]
    if len(pos_grp) < CONFIG["eval_min_positives"]:
        continue
    hold_rows = pos_grp.iloc[-CONFIG["eval_holdout_k"]:]
    cut_day = int(hold_rows.iloc[0]["day"])
    if cut_day <= 0:
        continue
    before_pos = grp[(grp["day"] < cut_day) & grp["is_rec"]]
    if len(before_pos) < CONFIG["eval_min_train_positives"]:
        # Skips accounts that dumped all reviews on a single day
        continue
    after_rows = grp[grp["day"] >= cut_day]
    hold_pos = set(int(x) for x in after_rows[after_rows["is_rec"]]["i_idx"].to_numpy())
    if len(hold_pos) >= 3:
        holdout_targets[u_int] = hold_pos
        drop_mask[after_rows.index.to_numpy()] = True
        if len(holdout_targets) >= CONFIG["eval_users"]:
            break

train_df = df[~drop_mask].reset_index(drop=True)
eval_user_list = sorted(holdout_targets.keys())
n_eval = len(eval_user_list)
assert n_eval >= 100, f"Too few valid chronological eval users: {n_eval}"

# Split 50/50 into DEV (tuning/model selection) and UNTOUCHED HOLDOUT TEST (final scorecard + 95% CI)
perm_eval = np.random.default_rng(SEED).permutation(n_eval)
dev_half = n_eval // 2
dev_rows = np.sort(perm_eval[:dev_half])
test_rows = np.sort(perm_eval[dev_half:])
print(
    f"Strict Chronological Split: {n_eval:,} users ({len(dev_rows)} DEV / {len(test_rows)} UNTOUCHED TEST) | "
    f"stripped {int(drop_mask.sum()):,} future rows"
)

# Per-game median playtime strictly on TRAIN positives (zero future leakage)
train_pos = train_df[train_df["is_rec"]]
med_series = train_pos.groupby("i_idx")["hours"].median()
median_hours = np.ones(n_items, dtype=np.float32) * 5.0
median_hours[med_series.index.to_numpy()] = np.clip(med_series.to_numpy(dtype=np.float32), 1.0, 200.0)

def build_weighted_matrix(
    frame: pd.DataFrame,
    med_h: np.ndarray,
    pos_base: float = 0.5,
    neg_weight: float = -0.35,
    salty_weight: float = 0.0,
) -> sp.csr_matrix:
    i_idx = frame["i_idx"].to_numpy()
    u_idx = frame["u_idx"].to_numpy()
    is_rec = frame["is_rec"].to_numpy()
    hrs = np.nan_to_num(frame["hours"].to_numpy(), nan=0.0, posinf=5000.0, neginf=0.0)
    m_h = med_h[i_idx]

    play_ratio = np.minimum(1.0, np.log1p(hrs) / np.log1p(m_h))
    pos_sig = pos_base + (1.0 - pos_base) * play_ratio

    salty_thresh = np.maximum(20.0, 1.5 * m_h)
    neg_sig = np.where(hrs <= salty_thresh, neg_weight, salty_weight)

    vals = np.where(is_rec, pos_sig, neg_sig).astype(np.float32)
    vals *= user_damping[u_idx]
    return sp.csr_matrix((vals, (u_idx, i_idx)), shape=(n_users, n_items), dtype=np.float32)

# Fast O(nnz_eval) construction of eval_seen_mask (replaces 1,000 full scans over 25M rows -> <0.05s!)
X_train_default = build_weighted_matrix(train_df, median_hours)
X_eval_hist = X_train_default[eval_user_list].tocsr()

eval_u_to_row = {u: r for r, u in enumerate(eval_user_list)}
eval_seen_mask = np.zeros((n_eval, n_items), dtype=bool)
eval_train_sub = train_df[train_df["u_idx"].isin(eval_u_to_row)]
for u, grp in eval_train_sub.groupby("u_idx", sort=False):
    eval_seen_mask[eval_u_to_row[int(u)], grp["i_idx"].to_numpy()] = True

train_item_pop = np.bincount(train_pos["i_idx"].to_numpy(), minlength=n_items).astype(np.float32)
DISCOUNTS_20 = 1.0 / np.log2(np.arange(2, 22, dtype=np.float64))

def evaluate_scores_matrix(score_mat: np.ndarray, label: str = "", subset_rows: np.ndarray | None = None, return_per_user: bool = False) -> dict:
    rows_sel = np.arange(n_eval) if subset_rows is None else subset_rows
    masked = score_mat[rows_sel].copy()
    masked[eval_seen_mask[rows_sel]] = -1e9

    k_top = min(20, n_items)
    top20_part = np.argpartition(-masked, k_top - 1, axis=1)[:, :k_top]
    r_idx = np.arange(len(rows_sel))[:, None]
    top20_scores = masked[r_idx, top20_part]
    order = np.argsort(-top20_scores, axis=1)
    top20 = top20_part[r_idx, order]

    ndcg10s, ndcg20s, rec20s = [], [], []
    unique_recs = set()
    for local_i, eval_r in enumerate(rows_sel):
        u = eval_user_list[int(eval_r)]
        targets = holdout_targets[u]
        preds = top20[local_i]
        unique_recs.update(int(x) for x in preds)
        hits = np.array([1.0 if int(p) in targets else 0.0 for p in preds], dtype=np.float64)

        k10 = min(10, len(targets))
        ndcg10s.append(float((hits[:10] * DISCOUNTS_20[:10]).sum() / DISCOUNTS_20[:k10].sum()))
        k20 = min(k_top, len(targets))
        ndcg20s.append(float((hits * DISCOUNTS_20[:k_top]).sum() / DISCOUNTS_20[:k20].sum()))
        rec20s.append(float(hits.sum() / len(targets)))

    out = {
        "label": label,
        "ndcg10": float(np.mean(ndcg10s)),
        "ndcg20": float(np.mean(ndcg20s)),
        "recall20": float(np.mean(rec20s)),
        "coverage20": len(unique_recs) / n_items,
    }
    if return_per_user:
        out["per_user_ndcg20"] = np.array(ndcg20s, dtype=np.float64)
        out["per_user_ndcg10"] = np.array(ndcg10s, dtype=np.float64)
    return out

pop_scores = np.broadcast_to(train_item_pop[None, :], (n_eval, n_items)).copy()
pop_dev = evaluate_scores_matrix(pop_scores, "Popularity (DEV)", subset_rows=dev_rows)
pop_test = evaluate_scores_matrix(pop_scores, "Popularity (TEST)", subset_rows=test_rows, return_per_user=True)
print(
    f"Popularity Baseline [DEV-500]  -> NDCG@10={pop_dev['ndcg10']:.4f} | NDCG@20={pop_dev['ndcg20']:.4f} | "
    f"Recall@20={pop_dev['recall20']:.4f} | Cov@20={pop_dev['coverage20']:.2%}"
)

In [ ]:
def compute_gram_gpu(X_csr: sp.csr_matrix, user_batch: int = 8192) -> np.ndarray:
    """
    Computes G = X^T X fast: sparse CPU BLAS (X.T @ X) is ~15-25s on High-RAM VMs and avoids
    T4 PCIe/dense-zero overhead (which took 125s on T4); on >=20GB L4/A100 uses GPU addmm_.
    """
    if VRAM_GB < 20.0:
        return (X_csr.T @ X_csr).toarray().astype(np.float32)
    n_u, n_i = X_csr.shape
    G_acc = torch.zeros((n_i, n_i), device=DEVICE, dtype=torch.float32)
    X_buf = torch.zeros((user_batch, n_i), device=DEVICE, dtype=torch.float32)
    for start in range(0, n_u, user_batch):
        end = min(start + user_batch, n_u)
        b_rows = end - start
        sub = X_csr[start:end]
        if sub.nnz == 0:
            continue
        coo = sub.tocoo()
        Xb = X_buf[:b_rows]
        Xb.zero_()
        r_t = torch.from_numpy(coo.row.astype(np.int64)).to(DEVICE)
        c_t = torch.from_numpy(coo.col.astype(np.int64)).to(DEVICE)
        v_t = torch.from_numpy(coo.data.astype(np.float32)).to(DEVICE)
        Xb.index_put_((r_t, c_t), v_t, accumulate=True)
        G_acc.addmm_(Xb.T, Xb)
        del r_t, c_t, v_t
    out = G_acc.cpu().numpy()
    del G_acc, X_buf
    torch.cuda.empty_cache()
    return out

def solve_ease_gpu(G_np: np.ndarray, reg_lambda: float, pop_alpha: float = 0.0) -> torch.Tensor:
    """
    Exact GPU EASE^R with symmetric Gram normalization AND source-side popularity de-amplification:
      G_tilde = D^-alpha G D^-alpha,  B_tilde = -P / diag(P)
      B_runtime = D^-alpha B_tilde  so browser `x @ B_runtime == (x D^-alpha) @ B_tilde`!
    """
    dtype = torch.float64 if USE_FP64_SOLVER else torch.float32
    G = torch.from_numpy(G_np).to(device=DEVICE, dtype=dtype)
    inv_pow = None
    if pop_alpha > 0.0:
        diag_g = torch.diagonal(G).clamp(min=1e-6)
        inv_pow = diag_g.pow(-pop_alpha)
        G.mul_(inv_pow.unsqueeze(1)).mul_(inv_pow.unsqueeze(0))
        scale = float(torch.diagonal(G).mean().item())
        eff_lambda = reg_lambda * (scale / max(1e-6, float(np.diag(G_np).mean())))
    else:
        eff_lambda = reg_lambda

    G.diagonal().add_(eff_lambda)
    L = torch.linalg.cholesky(G)
    P = torch.cholesky_inverse(L)
    del G, L
    diag_p = torch.diagonal(P).clamp(min=1e-12)
    B = -P / diag_p.unsqueeze(0)
    B.fill_diagonal_(0.0)
    if inv_pow is not None:
        # Cancel source-side d_i^+alpha amplification so CS2/GTA V don't drown out niche favorites!
        B.mul_(inv_pow.unsqueeze(1))
    del P
    torch.cuda.empty_cache()
    return B.to(torch.float32)

t0 = time.time()
G_train = compute_gram_gpu(X_train_default)
print(f"Computed Gram matrix G ({G_train.shape}) on GPU in {time.time() - t0:.2f}s")

X_eval_gpu = torch.from_numpy(X_eval_hist.toarray()).to(DEVICE, dtype=torch.float32)

# Tune (lambda, pop_alpha) strictly on DEV-500! (Includes higher lam=10000 & alpha=0.35 where NDCG was climbing)
SWEEP_LAMBDAS = [600.0, 1500.0, 3000.0, 6000.0, 10000.0]
SWEEP_ALPHAS = [0.15, 0.25, 0.35]

ease_results = []
best_ease = None
best_ease_scores = None
B_best_train_gpu = None

print(f"{'Config (DEV-500)':28s} | {'NDCG@10':>8s} | {'NDCG@20':>8s} | {'Rec@20':>8s} | {'Cov@20':>7s} | {'Time':>5s}")
print("-" * 80)
for alpha in SWEEP_ALPHAS:
    for lam in SWEEP_LAMBDAS:
        ts = time.time()
        B_gpu = solve_ease_gpu(G_train, reg_lambda=lam, pop_alpha=alpha)
        K = CONFIG["ease_k"]
        top_vals, top_idx = torch.topk(B_gpu, k=K, dim=1)
        B_sparse = torch.zeros_like(B_gpu).scatter_(1, top_idx, top_vals)
        scores_np = (X_eval_gpu @ B_sparse).cpu().numpy()
        del B_sparse, top_vals, top_idx

        label = f"EASE lam={int(lam):4d} a={alpha:.2f}"
        m = evaluate_scores_matrix(scores_np, label, subset_rows=dev_rows)
        m["lambda"] = lam
        m["alpha"] = alpha
        ease_results.append(m)
        print(f"{label:28s} | {m['ndcg10']:8.4f} | {m['ndcg20']:8.4f} | {m['recall20']:8.4f} | {m['coverage20']:6.2%} | {time.time()-ts:4.2f}s")
        if best_ease is None or m["ndcg20"] > best_ease["ndcg20"]:
            best_ease = m
            best_ease_scores = scores_np
            if B_best_train_gpu is not None:
                del B_best_train_gpu
            B_best_train_gpu = B_gpu
        else:
            del B_gpu
        torch.cuda.empty_cache()

print(f"\nWINNER EASE^R [DEV-500]: {best_ease['label']} -> NDCG@20={best_ease['ndcg20']:.4f} (vs Pop {pop_dev['ndcg20']:.4f}, +{100*(best_ease['ndcg20']/max(1e-9, pop_dev['ndcg20'])-1):.1f}%)")
assert best_ease["ndcg20"] > pop_dev["ndcg20"], "EASE^R must beat popularity baseline!"

# Check negative-review weight sensitivity on DEV-500 at winning (lambda, alpha)
for neg_w in [-0.15, 0.0]:
    X_alt = build_weighted_matrix(train_df, median_hours, neg_weight=neg_w)
    G_alt = compute_gram_gpu(X_alt)
    B_alt = solve_ease_gpu(G_alt, reg_lambda=best_ease["lambda"], pop_alpha=best_ease["alpha"])
    top_v, top_i = torch.topk(B_alt, k=CONFIG["ease_k"], dim=1)
    B_sp = torch.zeros_like(B_alt).scatter_(1, top_i, top_v)
    X_ev_alt = torch.from_numpy(X_alt[eval_user_list].toarray()).to(DEVICE, dtype=torch.float32)
    sc_alt = (X_ev_alt @ B_sp).cpu().numpy()
    m_alt = evaluate_scores_matrix(sc_alt, f"neg_w={neg_w}", subset_rows=dev_rows)
    print(f"  Signal check neg_w={neg_w:+.2f} [DEV-500]: NDCG@20={m_alt['ndcg20']:.4f} | Recall@20={m_alt['recall20']:.4f}")
    if m_alt["ndcg20"] > best_ease["ndcg20"]:
        best_ease.update(m_alt)
        best_ease["neg_weight"] = neg_w
        best_ease_scores = sc_alt
        X_train_default = X_alt
        X_eval_hist = X_alt[eval_user_list].tocsr()
        X_eval_gpu = X_ev_alt
        G_train = G_alt
        del B_best_train_gpu
        B_best_train_gpu = B_alt
    else:
        del B_alt
    del B_sp, top_v, top_i
    torch.cuda.empty_cache()

BEST_NEG_WEIGHT = best_ease.get("neg_weight", -0.35)
print(f"Selected neg_weight={BEST_NEG_WEIGHT}, lambda={best_ease['lambda']}, alpha={best_ease['alpha']}")

# Refit winning EASE^R on 100% of reviews (full data) and export CVEA v1
full_pos = df[df["is_rec"]]
full_med_s = full_pos.groupby("i_idx")["hours"].median()
full_median_hours = np.ones(n_items, dtype=np.float32) * 5.0
full_median_hours[full_med_s.index.to_numpy()] = np.clip(full_med_s.to_numpy(dtype=np.float32), 1.0, 200.0)

X_full = build_weighted_matrix(df, full_median_hours, neg_weight=BEST_NEG_WEIGHT)
G_full = compute_gram_gpu(X_full)
B_full_gpu = solve_ease_gpu(G_full, reg_lambda=best_ease["lambda"], pop_alpha=best_ease["alpha"])

K = CONFIG["ease_k"]
top_vals_full, top_idx_full = torch.topk(B_full_gpu, k=K, dim=1)
nbr_idx_u16 = top_idx_full.cpu().numpy().astype("<u2")
nbr_w_f16 = np.nan_to_num(top_vals_full.cpu().numpy(), nan=0.0, posinf=0.0, neginf=0.0).clip(-65000, 65000).astype("<f2")

EASE_OUT = OUT_DIR / f"game_ease_k{K}.cvea"
with open(EASE_OUT, "wb") as f:
    f.write(struct.pack("<4sIII", b"CVEA", 1, n_items, K))
    f.write(collab_igdb_ids.astype("<u4").tobytes())
    f.write(nbr_idx_u16.tobytes())
    f.write(nbr_w_f16.tobytes())

print(f"Exported {EASE_OUT.name}: {n_items:,} IGDB games x K={K} ({EASE_OUT.stat().st_size / (1024**2):.2f} MB)")
del G_full, top_vals_full, top_idx_full
gc.collect()
torch.cuda.empty_cache()

In [ ]:
DIM = CONFIG["collab_dim"]

def score_user_vecs_from_item_vecs(item_vecs: np.ndarray, X_hist_csr: sp.csr_matrix) -> np.ndarray:
    X_pos = X_hist_csr.copy()
    X_pos.data = np.maximum(X_pos.data, 0.0)
    u_vecs = X_pos @ item_vecs
    norms = np.linalg.norm(u_vecs, axis=1, keepdims=True)
    u_vecs = u_vecs / np.clip(norms, 1e-12, None)
    return (u_vecs @ item_vecs.T).astype(np.float32)

def apply_shrinkage(raw_vecs: np.ndarray, norm_power: float = 1.0, shrink: float = 0.25) -> np.ndarray:
    norms = np.linalg.norm(raw_vecs, axis=1, keepdims=True)
    denom = np.power(np.clip(norms, 1e-12, None), norm_power) + shrink
    return (raw_vecs / np.clip(denom, 1e-12, None)).astype(np.float32)

# --- Model 1: Popularity-Debiased Truncated SVD ---
def fit_svd_embeddings(X_mat: sp.csr_matrix, dim: int = 64, pop_exp: float = 0.25) -> np.ndarray:
    col_norms = np.sqrt(np.asarray(X_mat.power(2).sum(axis=0)).ravel()).clip(min=1e-6)
    inv_scale = np.power(col_norms, -pop_exp).astype(np.float32)
    X_scaled = X_mat @ sp.diags(inv_scale)
    eff_k = min(dim, min(X_scaled.shape) - 2)
    _, s, vt = svds(X_scaled, k=eff_k)
    order = np.argsort(s)[::-1]
    raw = (vt[order, :].T * np.sqrt(s[order])).astype(np.float32)
    if eff_k < dim:
        raw = np.pad(raw, ((0, 0), (0, dim - eff_k)))
    return raw

t0 = time.time()
svd_raw_train = fit_svd_embeddings(X_train_default, dim=DIM, pop_exp=0.25)
svd_vecs_train = apply_shrinkage(svd_raw_train, norm_power=1.0, shrink=0.20)
svd_scores = score_user_vecs_from_item_vecs(svd_vecs_train, X_eval_hist)
m_svd = evaluate_scores_matrix(svd_scores, "1. Debiased SVD-64", subset_rows=dev_rows)
print(f"{m_svd['label']:28s} [DEV] | NDCG@10={m_svd['ndcg10']:.4f} | NDCG@20={m_svd['ndcg20']:.4f} | Rec@20={m_svd['recall20']:.4f} | {time.time()-t0:.1f}s")

# --- Model 2: Positive-Eigenvalue Spectral EASE^R Embeddings (reuses cached B_best_train_gpu!) ---
# Uses torch.linalg.eigh to extract the top strictly POSITIVE algebraic eigenvalues (avoids svd_lowrank sign-flip on repulsive modes!)
def fit_spectral_ease_embeddings(B_gpu: torch.Tensor, dim: int = 64) -> np.ndarray:
    B_sym = 0.5 * (B_gpu + B_gpu.T)
    evals, evecs = torch.linalg.eigh(B_sym)
    k = min(dim, evals.numel())
    top_evals = evals[-k:].flip(0).clamp(min=0.0)
    top_evecs = evecs[:, -k:].flip(1)
    raw = (top_evecs * torch.sqrt(top_evals).unsqueeze(0)).cpu().numpy().astype(np.float32)
    del B_sym, evals, evecs, top_evals, top_evecs
    torch.cuda.empty_cache()
    if k < dim:
        raw = np.pad(raw, ((0, 0), (0, dim - k)))
    return raw

t0 = time.time()
spec_raw_train = fit_spectral_ease_embeddings(B_best_train_gpu, dim=DIM)
spec_vecs_train = apply_shrinkage(spec_raw_train, norm_power=1.0, shrink=0.15)
spec_scores = score_user_vecs_from_item_vecs(spec_vecs_train, X_eval_hist)
m_spec = evaluate_scores_matrix(spec_scores, "2. Spectral EASE-64", subset_rows=dev_rows)
print(f"{m_spec['label']:28s} [DEV] | NDCG@10={m_spec['ndcg10']:.4f} | NDCG@20={m_spec['ndcg20']:.4f} | Rec@20={m_spec['recall20']:.4f} | {time.time()-t0:.1f}s")
del B_best_train_gpu
torch.cuda.empty_cache()

# --- Model 3: GPU Conjugate-Gradient iALS with Rendle (2022) Frequency-Scaled Regularization ---
def fit_cg_ials_gpu(
    X_pos_csr: sp.csr_matrix,
    dim: int = 64,
    reg: float = 12.0,
    freq_exp: float = 0.5,
    alpha_conf: float = 12.0,
    n_iters: int = 12,
    cg_steps: int = 4,
) -> np.ndarray:
    coo = X_pos_csr.tocoo()
    mask = coo.data > 0
    rows = torch.from_numpy(coo.row[mask].astype(np.int64)).to(DEVICE)
    cols = torch.from_numpy(coo.col[mask].astype(np.int64)).to(DEVICE)
    w_edge = torch.from_numpy((alpha_conf * coo.data[mask]).astype(np.float32)).to(DEVICE)
    c_edge = 1.0 + w_edge
    n_u, n_i = X_pos_csr.shape

    # Frequency-scaled regularization prevents 40-review indie games from being crushed vs 300k-review hits!
    deg_u = torch.bincount(rows, minlength=n_u).to(torch.float32)
    deg_i = torch.bincount(cols, minlength=n_i).to(torch.float32)
    reg_u = reg * ((deg_u + 10.0) / deg_u.mean().clamp(min=1.0)).pow(freq_exp)
    reg_i = reg * ((deg_i + 10.0) / deg_i.mean().clamp(min=1.0)).pow(freq_exp)

    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    U = torch.randn((n_u, dim), device=DEVICE, dtype=torch.float32, generator=g) * 0.05
    V = torch.randn((n_i, dim), device=DEVICE, dtype=torch.float32, generator=g) * 0.05

    CHUNK = 2_000_000
    n_edges = rows.numel()

    def cg_solve(A_factors, B_factors, a_idx, b_idx, n_a, reg_vec):
        BtB = B_factors.T @ B_factors
        rhs = torch.zeros((n_a, dim), device=DEVICE, dtype=torch.float32)
        for s in range(0, n_edges, CHUNK):
            e = min(s + CHUNK, n_edges)
            rhs.index_add_(0, a_idx[s:e], B_factors[b_idx[s:e]] * c_edge[s:e].unsqueeze(1))

        def matvec(X):
            out = X @ BtB + reg_vec.unsqueeze(1) * X
            for s in range(0, n_edges, CHUNK):
                e = min(s + CHUNK, n_edges)
                ai = a_idx[s:e]
                bi = b_idx[s:e]
                dots = (X[ai] * B_factors[bi]).sum(dim=1) * w_edge[s:e]
                out.index_add_(0, ai, B_factors[bi] * dots.unsqueeze(1))
            return out

        X = A_factors
        r = rhs - matvec(X)
        p = r.clone()
        rs_old = (r * r).sum(dim=1, keepdim=True)
        for _ in range(cg_steps):
            Ap = matvec(p)
            denom = (p * Ap).sum(dim=1, keepdim=True).clamp(min=1e-10)
            step = rs_old / denom
            X = X + step * p
            r = r - step * Ap
            rs_new = (r * r).sum(dim=1, keepdim=True)
            p = r + (rs_new / rs_old.clamp(min=1e-10)) * p
            rs_old = rs_new
        return X

    for _ in range(n_iters):
        U = cg_solve(U, V, rows, cols, n_u, reg_u)
        V = cg_solve(V, U, cols, rows, n_i, reg_i)

    out = V.cpu().numpy().astype(np.float32)
    del U, V, rows, cols, w_edge, c_edge, deg_u, deg_i, reg_u, reg_i
    torch.cuda.empty_cache()
    return out

t0 = time.time()
ials_raw_train = fit_cg_ials_gpu(X_train_default, dim=DIM, reg=12.0, freq_exp=0.5, alpha_conf=12.0, n_iters=12)
ials_vecs_train = apply_shrinkage(ials_raw_train, norm_power=1.0, shrink=0.15)
ials_scores = score_user_vecs_from_item_vecs(ials_vecs_train, X_eval_hist)
m_ials = evaluate_scores_matrix(ials_scores, "3. Freq-Reg CG-iALS-64", subset_rows=dev_rows)
print(f"{m_ials['label']:28s} [DEV] | NDCG@10={m_ials['ndcg10']:.4f} | NDCG@20={m_ials['ndcg20']:.4f} | Rec@20={m_ials['recall20']:.4f} | {time.time()-t0:.1f}s")

def z_norm_rows(M: np.ndarray) -> np.ndarray:
    m = M.copy()
    m[eval_seen_mask] = np.nan
    mu = np.nanmean(m, axis=1, keepdims=True)
    sd = np.nanstd(m, axis=1, keepdims=True)
    out = (M - np.nan_to_num(mu)) / np.clip(np.nan_to_num(sd, nan=1.0), 1e-6, None)
    return out.astype(np.float32)

ease_z = z_norm_rows(best_ease_scores)
candidates = [
    ("svd", m_svd, svd_vecs_train, svd_scores),
    ("spectral_ease", m_spec, spec_vecs_train, spec_scores),
    ("cg_ials", m_ials, ials_vecs_train, ials_scores),
]
print("\n--- Blend check on DEV-500 (80% EASE^R + 20% Collab Vec + Standalone Embedding Quality) ---")
best_collab_name = None
best_collab_score = -1.0
best_collab_vecs_train = None
best_collab_scores_train = None

for name, m_single, vecs_tr, sc_tr in candidates:
    blend_sc = 0.80 * ease_z + 0.20 * z_norm_rows(sc_tr)
    mb = evaluate_scores_matrix(blend_sc, f"EASE + {name}", subset_rows=dev_rows)
    # Balance standalone 64-D geometry (personas/repulsion/Ridge alignment!) with blended lift
    combined_sel = 0.50 * m_single["ndcg20"] + 0.50 * mb["ndcg20"]
    print(
        f"  EASE + {name:14s} [DEV] | Standalone NDCG@20={m_single['ndcg20']:.4f} | "
        f"Blended(80/20) NDCG@20={mb['ndcg20']:.4f} | Rec@20={mb['recall20']:.4f} | Sel={combined_sel:.4f}"
    )
    if combined_sel > best_collab_score:
        best_collab_score = combined_sel
        best_collab_name = name
        best_collab_vecs_train = vecs_tr
        best_collab_scores_train = sc_tr

print(f"\nWINNER COLLAB MODEL [DEV-500]: {best_collab_name} (Selection score={best_collab_score:.4f})")

# Refit winning collaborative model on 100% of reviews and export CVEM v1
if best_collab_name == "svd":
    full_collab_raw = fit_svd_embeddings(X_full, dim=DIM, pop_exp=0.25)
    full_collab_vecs = apply_shrinkage(full_collab_raw, norm_power=1.0, shrink=0.20)
elif best_collab_name == "spectral_ease":
    full_collab_raw = fit_spectral_ease_embeddings(B_full_gpu, dim=DIM)
    full_collab_vecs = apply_shrinkage(full_collab_raw, norm_power=1.0, shrink=0.15)
else:
    full_collab_raw = fit_cg_ials_gpu(X_full, dim=DIM, reg=12.0, freq_exp=0.5, alpha_conf=12.0, n_iters=14)
    full_collab_vecs = apply_shrinkage(full_collab_raw, norm_power=1.0, shrink=0.15)

del B_full_gpu
torch.cuda.empty_cache()

COLLAB_OUT = OUT_DIR / "game_embeddings_f16.bin"
with open(COLLAB_OUT, "wb") as f:
    f.write(struct.pack("<4sIII", b"CVEM", 1, n_items, DIM))
    f.write(collab_igdb_ids.astype("<u4").tobytes())
    f.write(np.nan_to_num(full_collab_vecs, nan=0.0).astype("<f2").tobytes())

print(f"Exported {COLLAB_OUT.name}: {n_items:,} IGDB games x {DIM}-D ({COLLAB_OUT.stat().st_size / (1024**2):.2f} MB)")

In [ ]:
from sentence_transformers import SentenceTransformer

catalog_ids = np.array(sorted(igdb_catalog.keys()), dtype=np.uint32)
cat_id_to_row = {int(gid): i for i, gid in enumerate(catalog_ids)}

def format_game_card(g: dict, include_tags: bool = True) -> str:
    parts = [f"Title: {g['name']}" + (f" ({g['year']})" if g.get("year") else "")]
    if g.get("genres"):
        parts.append("Genres: " + ", ".join(g["genres"]))
    if g.get("themes"):
        parts.append("Themes: " + ", ".join(g["themes"]))
    if include_tags and g.get("steam_tags"):
        parts.append("Community Tags: " + ", ".join(g["steam_tags"]))
    if g.get("perspectives"):
        parts.append("Perspective: " + ", ".join(g["perspectives"]))
    if g.get("modes"):
        parts.append("Modes: " + ", ".join(g["modes"]))
    if g.get("keywords"):
        parts.append("Keywords: " + ", ".join(g["keywords"]))
    if g.get("franchises") or g.get("collections"):
        series = list(dict.fromkeys((g.get("franchises") or []) + (g.get("collections") or [])))
        parts.append("Series: " + ", ".join(series[:3]))
    if g.get("developers"):
        parts.append("Developer: " + ", ".join(g["developers"]))
    desc = g.get("summary") or g.get("steam_desc") or ""
    story = g.get("storyline") or ""
    if desc:
        parts.append("Summary: " + desc[:550])
    if story and story[:80] != desc[:80]:
        parts.append("Story: " + story[:320])
    return " | ".join(parts)

cards_full = [format_game_card(igdb_catalog[int(gid)], include_tags=True) for gid in catalog_ids]
matched_cat_rows = np.array([cat_id_to_row[int(gid)] for gid in collab_igdb_ids], dtype=np.int64)
# Second IGDB-only view (without Community Tags) for Tag-Dropout Ridge Augmentation!
cards_matched_notags = [format_game_card(igdb_catalog[int(gid)], include_tags=False) for gid in collab_igdb_ids]
print(f"Prepared {len(cards_full):,} full cards + {len(cards_matched_notags):,} tag-dropout cards (max_seq_length=512).")

t0 = time.time()
# Free large training matrices before loading BGE-large to keep RAM & T4 VRAM super lean!
for _tmp in ("df", "train_df", "train_pos", "full_pos", "X_train_default", "X_full", "G_train", "X_eval_gpu"):
    if _tmp in globals():
        del globals()[_tmp]
gc.collect()
torch.cuda.empty_cache()

model = SentenceTransformer(CONFIG["bge_model"], device="cuda")
model.half()  # Native FP16 weights (~670 MB vs 1.34 GB) — works across all transformers versions!
model.max_seq_length = 384 if VRAM_GB < 20.0 else 512  # OOM-proof on 14.6GB T4, full 512 on L4/A100

def encode_cards_fp16_safe(texts: list[str], desc_label: str) -> np.ndarray:
    # Encodes in 4,000-card chunks with safe batch_size so T4 VRAM stays <3GB and Colab CLI streams live progress!
    bs = 256 if VRAM_GB >= 20.0 else 64
    chunk_sz = 4000
    parts = []
    t_enc = time.time()
    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16):
        for s in range(0, len(texts), chunk_sz):
            e = min(s + chunk_sz, len(texts))
            emb_t = model.encode(
                texts[s:e],
                batch_size=bs,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_tensor=True,
            )
            parts.append(emb_t.float().cpu().numpy())
            del emb_t
            print(f"    [{desc_label}] {e:,}/{len(texts):,} encoded ({time.time() - t_enc:.1f}s)", flush=True)
    out = np.vstack(parts).astype(np.float32)
    del parts
    torch.cuda.empty_cache()
    print(f"  Encoded {desc_label}: {out.shape} in {time.time() - t_enc:.1f}s")
    return out

bge_1024 = encode_cards_fp16_safe(cards_full, "all catalog cards (with tags)")
E_matched_notags = encode_cards_fp16_safe(cards_matched_notags, "matched collab cards (IGDB-only tag-dropout)")
del model
torch.cuda.empty_cache()
print(f"Finished BGE-large encoding in {time.time() - t0:.1f}s")

# ==============================================================================
# Tag-Dropout Collaborative-Aligned Content Projection (1024-D -> 128-D):
#   Stacks [E_with_tags; E_igdb_only] -> [Y_collab; Y_collab] so W projects BOTH
#   tagged PC games AND tag-free console exclusives cleanly into collab space!
# ==============================================================================
E_matched = bge_1024[matched_cat_rows]
Y_collab_train = best_collab_vecs_train / np.clip(np.linalg.norm(best_collab_vecs_train, axis=1, keepdims=True), 1e-12, None)
Y_collab_full = full_collab_vecs / np.clip(np.linalg.norm(full_collab_vecs, axis=1, keepdims=True), 1e-12, None)

# 85/15 cold-start game split (evaluates BOTH tagged & untagged held-out cosine + Cold-Start Ranking NDCG!)
perm = np.random.default_rng(SEED).permutation(n_items)
split = int(0.85 * n_items)
tr_idx, va_idx = perm[:split], perm[split:]

E_tr_aug = torch.from_numpy(np.vstack([E_matched[tr_idx], E_matched_notags[tr_idx]])).to(DEVICE, dtype=torch.float64)
Y_tr_aug = torch.from_numpy(np.vstack([Y_collab_train[tr_idx], Y_collab_train[tr_idx]])).to(DEVICE, dtype=torch.float64)
E_va_tag = torch.from_numpy(E_matched[va_idx]).to(DEVICE, dtype=torch.float64)
E_va_not = torch.from_numpy(E_matched_notags[va_idx]).to(DEVICE, dtype=torch.float64)
Y_va = torch.from_numpy(Y_collab_train[va_idx]).to(DEVICE, dtype=torch.float64)

EtE = E_tr_aug.T @ E_tr_aug
EtY = E_tr_aug.T @ Y_tr_aug
I_1024 = torch.eye(bge_1024.shape[1], device=DEVICE, dtype=torch.float64)

best_gamma, best_va_cos = 30.0, -1.0
for gamma in [3.0, 10.0, 30.0, 80.0, 200.0, 400.0]:
    W = torch.linalg.solve(EtE + gamma * I_1024, EtY)
    p_tag = E_va_tag @ W
    p_tag = p_tag / p_tag.norm(dim=1, keepdim=True).clamp(min=1e-12)
    p_not = E_va_not @ W
    p_not = p_not / p_not.norm(dim=1, keepdim=True).clamp(min=1e-12)
    cos_tag = float((p_tag * Y_va).sum(dim=1).mean().item())
    cos_not = float((p_not * Y_va).sum(dim=1).mean().item())
    cos_avg = 0.5 * (cos_tag + cos_not)
    print(f"  Tag-Dropout Ridge gamma={gamma:5.1f} -> held-out cos (tagged={cos_tag:.4f}, IGDB-only={cos_not:.4f}, avg={cos_avg:.4f})")
    if cos_avg > best_va_cos:
        best_va_cos = cos_avg
        best_gamma = gamma

print(f"Selected gamma={best_gamma} (held-out avg cos={best_va_cos:.4f})")

# Next 64-D: Center + SVD PCA (mild whitening s^-0.25 preserves niche genres)
E_all_gpu = torch.from_numpy(bge_1024).to(DEVICE, dtype=torch.float32)
E_centered = E_all_gpu - E_all_gpu.mean(dim=0, keepdim=True)
_, S_pca, V_pca = torch.linalg.svd(E_centered, full_matrices=False)
pca_proj = V_pca[:64].T * S_pca[:64].pow(-0.25).unsqueeze(0)
sem_64 = E_centered @ pca_proj
sem_64 = sem_64 / sem_64.norm(dim=1, keepdim=True).clamp(min=1e-12)

# True Cold-Start Ranking Benchmark on the 15% strictly unseen cold-start games (`va_idx`, fitted ONLY on `tr_idx`!)
W_cold_eval = torch.linalg.solve(EtE + best_gamma * I_1024, EtY).to(torch.float32)
al_cold = E_all_gpu @ W_cold_eval
al_cold = al_cold / al_cold.norm(dim=1, keepdim=True).clamp(min=1e-12)

va_set = set(int(x) for x in va_idx)
cold_eval_users = [r for r, u in enumerate(eval_user_list) if len(holdout_targets[u] & va_set) >= 1]
X_hist_cold = X_eval_hist.copy()
# Mask held-out 15% games from user history too so va_idx items are 100% cold-start!
va_mask_cols = np.zeros(n_items, dtype=bool)
va_mask_cols[va_idx] = True
X_hist_cold_csc = X_hist_cold.tolil()
X_hist_cold_csc[:, va_idx] = 0.0
X_hist_cold = X_hist_cold_csc.tocsr()

def eval_cold_subset(w_al_val: float) -> tuple[float, float]:
    w_s = math.sqrt(max(0.0, 1.0 - w_al_val * w_al_val))
    c_vec = torch.cat([al_cold * w_al_val, sem_64 * w_s], dim=1)
    c_vec = (c_vec / c_vec.norm(dim=1, keepdim=True).clamp(min=1e-12))[matched_cat_rows].cpu().numpy()
    sc = score_user_vecs_from_item_vecs(c_vec, X_hist_cold)
    # Rank strictly among the 15% cold-start catalog (simulating Bucket E-Cold / unindexed releases!)
    non_cold = ~va_mask_cols
    sc[:, non_cold] = -1e9
    nd20, rc20 = [], []
    for r in cold_eval_users:
        u = eval_user_list[r]
        t_cold = holdout_targets[u] & va_set
        row_sc = sc[r].copy()
        row_sc[eval_seen_mask[r]] = -1e9
        preds = np.argsort(-row_sc)[:20]
        hits = np.array([1.0 if int(p) in t_cold else 0.0 for p in preds], dtype=np.float64)
        idcg = DISCOUNTS_20[: min(20, len(t_cold))].sum()
        nd20.append(float((hits * DISCOUNTS_20).sum() / idcg))
        rc20.append(float(hits.sum() / len(t_cold)))
    return (float(np.mean(nd20)) if nd20 else 0.0, float(np.mean(rc20)) if rc20 else 0.0)

cold_pca_ndcg20, cold_pca_rec20 = eval_cold_subset(0.0)
print(f"\n--- True Cold-Start Game Benchmark ({len(va_idx):,} unseen games, {len(cold_eval_users)} users) ---")
print(f"  Pure BGE-PCA (w_aligned=0.00) -> Cold NDCG@20={cold_pca_ndcg20:.4f} | Cold Recall@20={cold_pca_rec20:.4f}")

# Fit leak-free train-split alignment across all matched items to tune w_aligned on DEV-500
E_all_aug = torch.from_numpy(np.vstack([E_matched, E_matched_notags])).to(DEVICE, dtype=torch.float64)
Y_tr_all_aug = torch.from_numpy(np.vstack([Y_collab_train, Y_collab_train])).to(DEVICE, dtype=torch.float64)
W_train = torch.linalg.solve(E_all_aug.T @ E_all_aug + best_gamma * I_1024, E_all_aug.T @ Y_tr_all_aug).to(torch.float32)
aligned_64_train = E_all_gpu @ W_train
aligned_64_train = aligned_64_train / aligned_64_train.norm(dim=1, keepdim=True).clamp(min=1e-12)

best_mix_w, best_content_ndcg, best_content_scores = 0.65, -1.0, None
best_cold_ndcg20, best_cold_rec20 = cold_pca_ndcg20, cold_pca_rec20
for w_al in [0.0, 0.40, 0.55, 0.65, 0.75, 0.85]:
    w_sem = math.sqrt(max(0.0, 1.0 - w_al * w_al))
    comb = torch.cat([aligned_64_train * w_al, sem_64 * w_sem], dim=1)
    comb = comb / comb.norm(dim=1, keepdim=True).clamp(min=1e-12)
    comb_matched = comb[matched_cat_rows].cpu().numpy()
    sc = score_user_vecs_from_item_vecs(comb_matched, X_eval_hist)
    mc = evaluate_scores_matrix(sc, f"Content w_al={w_al:.2f}", subset_rows=dev_rows)
    c_nd20, c_rc20 = eval_cold_subset(w_al) if w_al > 0 else (cold_pca_ndcg20, cold_pca_rec20)
    print(f"  w_aligned={w_al:.2f} | Warm DEV NDCG@20={mc['ndcg20']:.4f} | True Cold NDCG@20={c_nd20:.4f} (Rec@20={c_rc20:.4f})")
    if mc["ndcg20"] > best_content_ndcg:
        best_content_ndcg = mc["ndcg20"]
        best_mix_w = w_al
        best_content_scores = sc
        best_cold_ndcg20, best_cold_rec20 = c_nd20, c_rc20

cold_lift_pct = 100.0 * (best_cold_ndcg20 / max(1e-9, cold_pca_ndcg20) - 1.0)
print(f"Selected w_aligned={best_mix_w:.2f} (True Cold-Start NDCG@20 lift vs pure BGE-PCA: {cold_lift_pct:+.1f}%)")

# Final export: refit Tag-Dropout Ridge alignment on 100% full collaborative vectors -> project ALL IGDB games!
Y_full_all_aug = torch.from_numpy(np.vstack([Y_collab_full, Y_collab_full])).to(DEVICE, dtype=torch.float64)
W_full = torch.linalg.solve(E_all_aug.T @ E_all_aug + best_gamma * I_1024, E_all_aug.T @ Y_full_all_aug).to(torch.float32)
aligned_64_full = E_all_gpu @ W_full
aligned_64_full = aligned_64_full / aligned_64_full.norm(dim=1, keepdim=True).clamp(min=1e-12)

w_sem_best = math.sqrt(max(0.0, 1.0 - best_mix_w * best_mix_w))
content_128 = torch.cat([aligned_64_full * best_mix_w, sem_64 * w_sem_best], dim=1)
content_128 = (content_128 / content_128.norm(dim=1, keepdim=True).clamp(min=1e-12)).cpu().numpy().astype(np.float32)

CONTENT_OUT = OUT_DIR / "game_content_f16.bin"
with open(CONTENT_OUT, "wb") as f:
    f.write(struct.pack("<4sIII", b"CVEM", 1, len(catalog_ids), CONFIG["content_dim"]))
    f.write(catalog_ids.astype("<u4").tobytes())
    f.write(np.nan_to_num(content_128, nan=0.0).astype("<f2").tobytes())

print(f"\nExported {CONTENT_OUT.name}: {len(catalog_ids):,} IGDB games x 128-D ({CONTENT_OUT.stat().st_size / (1024**2):.2f} MB)")

print("\n=== QUALITATIVE COLD-START & NEIGHBOR SANITY CHECK ===")
for probe_name in [
    "Bloodborne",
    "The Legend of Zelda: Tears of the Kingdom",
    "Astro Bot",
    "Black Myth: Wukong",
    "Balatro",
    "Elden Ring",
    "Hollow Knight",
    "Hades",
]:
    matches = [gid for gid, g in igdb_catalog.items() if g["name"].lower() == probe_name.lower()]
    if not matches:
        continue
    gid = matches[0]
    row = cat_id_to_row[gid]
    sims = content_128 @ content_128[row]
    top_n = np.argsort(-sims)[1:7]
    nbr_str = ", ".join(f"{igdb_catalog[int(catalog_ids[j])]['name']} ({sims[j]:.2f})" for j in top_n)
    src = "COLLAB+CONTENT" if gid in igdb_to_idx else "COLD-START ONLY"
    print(f"  [{src:15s}] {probe_name} -> {nbr_str}")

In [ ]:
# 1. Tune 4-way fusion weights strictly on DEV-500, then verify once on UNTOUCHED TEST-500!
collab_z = z_norm_rows(best_collab_scores_train)
content_z = z_norm_rows(best_content_scores)
pop_log = np.log1p(train_item_pop)
pop_z = ((pop_log - pop_log.mean()) / max(1e-6, pop_log.std())).astype(np.float32)[None, :]

best_fusion_dev = None
best_fusion_scores = None
for w_ease in [0.50, 0.60, 0.70, 0.80]:
    for w_col in [0.10, 0.20, 0.30]:
        for w_cnt in [0.05, 0.10, 0.15, 0.20]:
            for w_pop in [0.0, 0.03, 0.06]:
                tot = w_ease + w_col + w_cnt + w_pop
                sc = (w_ease * ease_z + w_col * collab_z + w_cnt * content_z + w_pop * pop_z) / tot
                mf_dev = evaluate_scores_matrix(sc, "fusion_dev", subset_rows=dev_rows)
                if best_fusion_dev is None or mf_dev["ndcg20"] > best_fusion_dev["ndcg20"]:
                    best_fusion_dev = {
                        **mf_dev,
                        "w_ease": round(w_ease / tot, 4),
                        "w_collab": round(w_col / tot, 4),
                        "w_content": round(w_cnt / tot, 4),
                        "w_pop": round(w_pop / tot, 4),
                    }
                    best_fusion_scores = sc

# Untouched Holdout-500 Verification + Paired Bootstrap 95% CI (1,000 resamples)
ease_test = evaluate_scores_matrix(best_ease_scores, "EASE-only (TEST)", subset_rows=test_rows, return_per_user=True)
fusion_test = evaluate_scores_matrix(best_fusion_scores, "Fusion (TEST)", subset_rows=test_rows, return_per_user=True)

def paired_bootstrap_ci(a: np.ndarray, b: np.ndarray, n_boot: int = 1000, seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    diffs = a - b
    idx = rng.integers(0, len(diffs), size=(n_boot, len(diffs)))
    means = diffs[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return {
        "meanDiff": round(float(diffs.mean()), 5),
        "ci95": [round(float(lo), 5), round(float(hi), 5)],
        "significantPositive": bool(lo > 0),
    }

ci_vs_pop = paired_bootstrap_ci(fusion_test["per_user_ndcg20"], pop_test["per_user_ndcg20"], seed=SEED)
ci_vs_ease = paired_bootstrap_ci(fusion_test["per_user_ndcg20"], ease_test["per_user_ndcg20"], seed=SEED)

print("=== UNTOUCHED HOLDOUT TEST-500 SCORECARD ===")
print(f"  Popularity [TEST] -> NDCG@10={pop_test['ndcg10']:.4f} | NDCG@20={pop_test['ndcg20']:.4f} | Rec@20={pop_test['recall20']:.4f}")
print(f"  EASE^R     [TEST] -> NDCG@10={ease_test['ndcg10']:.4f} | NDCG@20={ease_test['ndcg20']:.4f} | Rec@20={ease_test['recall20']:.4f} | Cov@20={ease_test['coverage20']:.2%}")
print(f"  Full Fusion[TEST] -> NDCG@10={fusion_test['ndcg10']:.4f} | NDCG@20={fusion_test['ndcg20']:.4f} | Rec@20={fusion_test['recall20']:.4f} | Cov@20={fusion_test['coverage20']:.2%}")
print(f"  95% Bootstrap CI (Fusion - Pop  NDCG@20): {ci_vs_pop}")
print(f"  95% Bootstrap CI (Fusion - EASE NDCG@20): {ci_vs_ease}")
print(f"  Learned blend weights: ease={best_fusion_dev['w_ease']} collab={best_fusion_dev['w_collab']} content={best_fusion_dev['w_content']} pop={best_fusion_dev['w_pop']}")

metric_keys = ("ndcg10", "ndcg20", "recall20", "coverage20")
ranker_payload = {
    "_source": "colab-gpu-validated-v1.1",
    "dataset": "antonkozyriev/game-recommendations-on-steam + FronkonGames/atalaydenknalbant 2026 + IGDB v4",
    "collab_model": best_collab_name,
    "ease_k": CONFIG["ease_k"],
    "ease_lambda": best_ease["lambda"],
    "ease_pop_alpha": best_ease["alpha"],
    "ease_neg_weight": BEST_NEG_WEIGHT,
    "content_model": CONFIG["bge_model"],
    "content_align_gamma": best_gamma,
    "content_align_weight": best_mix_w,
    "blend": {
        "ease": best_fusion_dev["w_ease"],
        "collab": best_fusion_dev["w_collab"],
        "content": best_fusion_dev["w_content"],
        "popularityPrior": best_fusion_dev["w_pop"],
    },
    "metrics": {
        "devUsers": int(len(dev_rows)),
        "untouchedTestUsers": int(len(test_rows)),
        "dev": {
            "popularity": {k: pop_dev[k] for k in metric_keys},
            "easeOnly": {k: best_ease[k] for k in metric_keys},
            "fusion": {k: best_fusion_dev[k] for k in metric_keys},
        },
        "untouchedTest": {
            "popularity": {k: pop_test[k] for k in metric_keys},
            "easeOnly": {k: ease_test[k] for k in metric_keys},
            "fusion": {k: fusion_test[k] for k in metric_keys},
            "bootstrapFusionMinusPopNdcg20": ci_vs_pop,
            "bootstrapFusionMinusEaseNdcg20": ci_vs_ease,
        },
        "trueColdStart": {
            "heldoutColdGames": int(len(va_idx)),
            "pureBgePcaNdcg20": round(cold_pca_ndcg20, 5),
            "alignedHybridNdcg20": round(best_cold_ndcg20, 5),
            "alignedHybridRecall20": round(best_cold_rec20, 5),
            "relativeLiftPct": round(cold_lift_pct, 2),
        },
    },
}
WEIGHTS_OUT = OUT_DIR / "game_ranker_weights.json"
WEIGHTS_OUT.write_text(json.dumps(ranker_payload, indent=2), encoding="utf-8")

# 2. Distill & Audit 128-D Game Taste Tuner Axes (Intensity/Pacing & Tone), orthogonalized against popularity!
def auc_score(pos_s, neg_s):
    wins = sum(1.0 if p > n else (0.5 if p == n else 0.0) for p in pos_s for n in neg_s)
    return wins / max(1, len(pos_s) * len(neg_s))

cat_log_pop = np.array([math.log1p(igdb_catalog[int(gid)]["total_rating_count"]) for gid in catalog_ids], dtype=np.float32)
# Compute popularity direction in content_128 space so mood sliders never accidentally bias toward popularity
pop_centered = cat_log_pop - cat_log_pop.mean()
pop_dir = content_128.T @ pop_centered
pop_dir = pop_dir / max(1e-12, float(np.linalg.norm(pop_dir)))

def build_axis(pos_vecs, neg_vecs, seed=42):
    assert len(pos_vecs) >= 12 and len(neg_vecs) >= 12, f"Insufficient mood anchors: pos={len(pos_vecs)}, neg={len(neg_vecs)}"
    rng = random.Random(seed)
    p_idx = list(range(len(pos_vecs)))
    n_idx = list(range(len(neg_vecs)))
    rng.shuffle(p_idx)
    rng.shuffle(n_idx)
    ps = max(8, int(0.8 * len(p_idx)))
    ns = max(8, int(0.8 * len(n_idx)))
    axis = np.mean([pos_vecs[i] for i in p_idx[:ps]], axis=0) - np.mean([neg_vecs[i] for i in n_idx[:ns]], axis=0)
    # Orthogonalize out global popularity direction so sliders measure pure mood/pacing
    axis = axis - float(np.dot(axis, pop_dir)) * pop_dir
    axis = axis / max(1e-12, float(np.linalg.norm(axis)))
    vp = [float(np.dot(pos_vecs[i], axis)) for i in p_idx[ps:]]
    vn = [float(np.dot(neg_vecs[i], axis)) for i in n_idx[ns:]]
    return axis.astype(np.float32), auc_score(vp, vn)

high_int, low_int = [], []
warm_tone, dark_tone = [], []

sorted_cat = sorted(igdb_catalog.values(), key=lambda x: x["total_rating_count"], reverse=True)
for g in sorted_cat:
    if g["total_rating_count"] < 20:
        continue
    row = cat_id_to_row[g["id"]]
    vec = content_128[row]
    all_lbl = {x.lower() for x in (g["genres"] + g["themes"] + g.get("steam_tags", []))}

    if (all_lbl & {"shooter", "fighting", "hack and slash/beat 'em up", "fps", "fast-paced", "boomer shooter", "bullet hell"}) and not (
        all_lbl & {"visual novel", "point-and-click", "turn-based strategy (tbs)", "cozy", "relaxing"}
    ):
        if len(high_int) < 150:
            high_int.append(vec)
    elif (all_lbl & {"visual novel", "point-and-click", "cozy", "relaxing", "puzzle", "card & board game", "farming sim"}) and not (
        all_lbl & {"shooter", "action", "fighting", "horror", "hack and slash/beat 'em up"}
    ):
        if len(low_int) < 150:
            low_int.append(vec)

    if (all_lbl & {"comedy", "kids", "cozy", "cute", "family friendly", "wholesome", "colorful"}) and not (
        all_lbl & {"horror", "survival", "dark fantasy", "psychological horror", "gore", "warfare", "survival horror"}
    ):
        if len(warm_tone) < 150:
            warm_tone.append(vec)
    elif (all_lbl & {"horror", "psychological horror", "dark fantasy", "survival horror", "gore", "lovecraftian"}) and not (
        all_lbl & {"comedy", "kids", "cozy", "cute", "family friendly", "wholesome"}
    ):
        if len(dark_tone) < 150:
            dark_tone.append(vec)

intensity_arr, int_auc = build_axis(high_int, low_int, seed=SEED)
tone_arr, tone_auc = build_axis(warm_tone, dark_tone, seed=SEED)
intensity_vec = [round(float(x), 5) for x in intensity_arr]
tone_vec = [round(float(x), 5) for x in tone_arr]

# Export BOTH pacingVec (for drop-in compatibility with src/services/tunerService.js!) AND intensityVec
TUNER_OUT = OUT_DIR / "game_tuner_axes.json"
TUNER_OUT.write_text(
    json.dumps(
        {
            "version": 1,
            "dim": CONFIG["content_dim"],
            "pacingVec": intensity_vec,
            "intensityVec": intensity_vec,
            "toneVec": tone_vec,
        },
        separators=(",", ":"),
    ),
    encoding="utf-8",
)
print(f"\nExported {TUNER_OUT.name} | Intensity AUC={int_auc:.3f} ({len(high_int)}/{len(low_int)}) | Tone AUC={tone_auc:.3f} ({len(warm_tone)}/{len(dark_tone)})")

# Audit correlation with popularity + print Top-5 / Bottom-5 well-known games on each axis!
well_known_mask = np.array([igdb_catalog[int(gid)]["total_rating_count"] >= 150 for gid in catalog_ids])
wk_indices = np.where(well_known_mask)[0]
for axis_name, axis_vec in [("Intensity (+Action / -Cozy)", intensity_arr), ("Tone (+Warm / -Dark)", tone_arr)]:
    proj = content_128 @ axis_vec
    corr_pop = float(np.corrcoef(proj, cat_log_pop)[0, 1])
    wk_order = wk_indices[np.argsort(-proj[wk_indices])]
    top5 = [igdb_catalog[int(catalog_ids[j])]["name"] for j in wk_order[:5]]
    bot5 = [igdb_catalog[int(catalog_ids[j])]["name"] for j in wk_order[-5:][::-1]]
    print(f"  {axis_name:28s} | corr(axis, log_pop)={corr_pop:+.3f}")
    print(f"    + Pole: {', '.join(top5)}")
    print(f"    - Pole: {', '.join(bot5)}")

In [ ]:
# Export compact Steam app_id -> IGDB id crosswalk for instant 0-API Steam Library imports in Phase 2!
STEAM_MAP_OUT = OUT_DIR / "game_steam_to_igdb.json"
STEAM_MAP_OUT.write_text(
    json.dumps({str(k): int(v) for k, v in sorted(steam_to_igdb.items())}, separators=(",", ":")),
    encoding="utf-8",
)

# Exact byte-level verification matching CinemaVault's browser parsers (easeService.js, embeddingService.js, tunerService.js)
def verify_cvea(path: Path):
    buf = path.read_bytes()
    magic, ver, count, k = struct.unpack("<4sIII", buf[:16])
    expected = 16 + count * 4 + count * k * 4
    assert magic == b"CVEA" and ver == 1, f"Bad CVEA header: {magic} v{ver}"
    assert len(buf) == expected, f"Bad CVEA length: {len(buf)} != {expected}"
    assert 0 < count <= 65535 and 0 < k <= 512
    ids = np.frombuffer(buf, dtype="<u4", count=count, offset=16)
    nbr_idx = np.frombuffer(buf, dtype="<u2", count=count * k, offset=16 + count * 4)
    nbr_w = np.frombuffer(buf, dtype="<f2", count=count * k, offset=16 + count * 4 + count * k * 2)
    assert len(np.unique(ids)) == count, "Duplicate IDs in CVEA!"
    assert int(nbr_idx.max()) < count, "Out-of-bounds neighbor index in CVEA!"
    assert np.all(np.isfinite(nbr_w)), "Non-finite float16 weights in CVEA!"
    print(f"  [OK] {path.name:28s} | CVEA v1 | {count:6,} games x K={k:<3d} | {len(buf)/(1024**2):.2f} MB")

def verify_cvem(path: Path):
    buf = path.read_bytes()
    magic, ver, count, dim = struct.unpack("<4sIII", buf[:16])
    expected = 16 + count * 4 + count * dim * 2
    assert magic == b"CVEM" and ver == 1, f"Bad CVEM header: {magic} v{ver}"
    assert len(buf) == expected, f"Bad CVEM length: {len(buf)} != {expected}"
    assert 0 < count <= 500000 and 0 < dim <= 1024
    ids = np.frombuffer(buf, dtype="<u4", count=count, offset=16)
    vecs = np.frombuffer(buf, dtype="<f2", count=count * dim, offset=16 + count * 4)
    assert len(np.unique(ids)) == count, "Duplicate IDs in CVEM!"
    assert np.all(np.isfinite(vecs)), "Non-finite floats in CVEM!"
    print(f"  [OK] {path.name:28s} | CVEM v1 | {count:6,} games x {dim:3d}-D | {len(buf)/(1024**2):.2f} MB")

print("=== BINARY & JSON VERIFICATION ===")
verify_cvea(EASE_OUT)
verify_cvem(COLLAB_OUT)
verify_cvem(CONTENT_OUT)
tuner_parsed = json.loads(TUNER_OUT.read_text(encoding="utf-8"))
assert isinstance(tuner_parsed.get("pacingVec"), list) and isinstance(tuner_parsed.get("toneVec"), list)
for jf in [WEIGHTS_OUT, TUNER_OUT, STEAM_MAP_OUT]:
    json.loads(jf.read_text(encoding="utf-8"))
    print(f"  [OK] {jf.name:28s} | JSON    | {jf.stat().st_size / 1024:.1f} KB")

BUNDLE_PATH = Path("/content/game_models_bundle.zip")
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.iterdir()):
        zf.write(p, arcname=p.name)

total_mb = sum(p.stat().st_size for p in OUT_DIR.iterdir()) / (1024 ** 2)
print(f"\nBundle ready: {BUNDLE_PATH} ({BUNDLE_PATH.stat().st_size / (1024**2):.2f} MB zipped | {total_mb:.2f} MB unzipped)")
print("Unzip all files into: public/models/games/")

try:
    from google.colab import files
    files.download(str(BUNDLE_PATH))
except Exception as e:
    print(f"(Auto-download skipped outside Colab UI: {e})")